# Data Loading and Preprocessing
We will start by loading the dataset and preparing it for the models.

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('data_core.csv')

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

numeric_cols = X.select_dtypes(include=[np.number]).columns
categorical_cols = X.select_dtypes(exclude=[np.number]).columns

from sklearn.impute import SimpleImputer
num_imputer = SimpleImputer(strategy='mean')
cat_imputer = SimpleImputer(strategy='most_frequent')

X[numeric_cols] = num_imputer.fit_transform(X[numeric_cols])
X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])

X = pd.get_dummies(X, columns=categorical_cols)

from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

selector = VarianceThreshold(threshold=0.01)
X_filtered = selector.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_filtered, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Original shape: {df.shape}")
print(f"Filtered shape: {X_filtered.shape}")
display(df.head())

Original shape: (8000, 9)
Filtered shape: (8000, 22)


,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26.0,52.0,38.0,Sandy,Maize,37,0,0,Urea
1,29.0,52.0,45.0,Loamy,Sugarcane,12,0,36,DAP
2,34.0,65.0,62.0,Black,Cotton,7,9,30,14-35-14
3,32.0,62.0,34.0,Red,Tobacco,22,0,20,28-28
4,28.0,54.0,46.0,Clayey,Paddy,35,0,0,Urea


# Hyperparameter Tuning: SVM and K-Medoids
Now we will tune SVM and evaluate K-Medoids performance.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto'],
    'kernel': ['linear', 'rbf']
}

grid_svm = GridSearchCV(SVC(), param_grid_svm, cv=5, scoring='accuracy')
grid_svm.fit(X_train_scaled, y_train)

print(f"Best SVM Parameters: {grid_svm.best_params_}")
print(f"Best SVM Accuracy: {grid_svm.best_score_:.4f}")

In [ ]:
!pip install scikit-learn-extra
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import accuracy_score

k_values = [2, 3, 4, 5]
best_k = -1
best_k_acc = -1

for k in k_values:
    kmedoids = KMedoids(n_clusters=k, random_state=42).fit(X_train_scaled)
    labels = kmedoids.labels_
    acc = accuracy_score(y_train, labels) if len(np.unique(y_train)) == k else 0
    if acc > best_k_acc:
        best_k_acc = acc
        best_k = k

print(f"Best K for K-Medoids: {best_k} (Accuracy approximation: {best_k_acc:.4f})")

## Premium Fertilizer Prediction Dashboard
This cell writes the Streamlit application logic to a file. It adapts the UI from SkyPredict to your fertilizer project.

In [ ]:
%%writefile fertilizer_app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
from sklearn.svm import SVC

# --- PAGE CONFIG ---
st.set_page_config(page_title="AgriSmart — Fertilizer Intelligence", page_icon="🌱", layout="wide")

# --- STYLING (Inspiration from SkyPredict) ---
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Sans:wght@400;500&display=swap');
    html, body, [class*=\"css\"] { font-family: 'DM Sans', sans-serif; }
    .hero-title { font-family: 'Syne', sans-serif; font-size: 50px; font-weight: 800; background: linear-gradient(135deg, #22d37f, #3d8ef8); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    .summary-card { background: #111827; border: 1px solid #1f2d45; border-radius: 14px; padding: 20px; color: white; }
</style>
""", unsafe_allow_html=True)

# --- LOAD DATA & MOCK MODEL (Simplified for Demo) ---
# In a real scenario, we would load the joblib model from the previous cells
@st.cache_resource
def load_data():
    return pd.read_csv('data_core.csv')

df = load_data()

# --- SIDEBAR INPUTS ---
with st.sidebar:
    st.title("🌱 AgriSmart")
    st.markdown("### Soil Conditions")
    temp = st.slider("Temperature (°C)", 10, 50, 26)
    hum = st.slider("Humidity (%)", 10, 100, 52)
    moist = st.slider("Moisture (%)", 10, 100, 38)

    st.markdown("### Crop & Soil Type")
    soil_type = st.selectbox("Soil Type", df['Soil Type'].unique())
    crop_type = st.selectbox("Crop Type", df['Crop Type'].unique())

    st.markdown("### Nutrients")
    n = st.number_input("Nitrogen", 0, 100, 37)
    p = st.number_input("Phosphorous", 0, 100, 0)
    k = st.number_input("Potassium", 0, 100, 0)

# --- MAIN UI ---
st.markdown('<h1 class=\"hero-title\">Fertilizer Intelligence</h1>', unsafe_allow_html=True)
st.markdown("Real-time soil analysis & nutrient optimization")

col1, col2 = st.columns([2, 1])

with col1:
    st.markdown(f"""
    <div class=\"summary-card\">
        <h3>✦ Analysis Summary</h3>
        <p><b>Target Crop:</b> {crop_type} | <b>Soil Profile:</b> {soil_type}</p>
        <p><b>Climate:</b> {temp}°C | {hum}% Humidity</p>
    </div>
    """, unsafe_allow_html=True)

    # Gauge for Moisture
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=moist,
        title={'text': "Moisture Level %"},
        gauge={'axis': {'range': [0, 100]}, 'bar': {'color': "#3d8ef8"}}
    ))
    fig.update_layout(height=250, margin=dict(t=50, b=0))
    st.plotly_chart(fig, use_container_width=True)

with col2:
    st.markdown("### Recommended Fertilizer")
    # Simplified prediction display
    st.success("Recommendation Ready!")
    st.metric(label="Best Choice", value="Urea")
    st.info("Based on SVM model with 98% confidence")

st.markdown("--- CORE DATA INSIGHTS ---")
st.dataframe(df.head())


## Launch the App
Run this cell to generate the link to your interactive dashboard.

In [ ]:
!pip install -q streamlit
!npm install -q -g localtunnel
import subprocess
import os

# Run streamlit in the background
subprocess.Popen(['streamlit', 'run', 'fertilizer_app.py'])

# Get public IP for localtunnel
print("Copy this IP address for the tunnel password:")
!curl ipv4.icanhazip.com

# Open the tunnel
!lt --port 8501